<a href="https://colab.research.google.com/github/Tushika2024/MultiModal_AI/blob/main/0_datasetprep_audio_frame_embeddings.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_DIR = "/content/drive/MyDrive/multimodal_project/dataset"

os.chdir(PROJECT_DIR)

print(os.getcwd())

In [ ]:
!ls -lh

In [ ]:
!unzip -q archive.zip

In [ ]:
#after extraction to check folder structure
import os

for item in os.listdir():
    print(item)

In [ ]:
##checking videos in full datset
import os

video_dir = "/content/drive/MyDrive/multimodal_project/dataset/TrainValVideo"

videos = os.listdir(video_dir)

print("Number of videos:", len(videos))
print(videos[:10])

In [ ]:
#creating a subst
import os
import shutil

video_dir = "/content/drive/MyDrive/multimodal_project/dataset/TrainValVideo"

subset_dir = "subset_videos"

os.makedirs(subset_dir, exist_ok=True)

videos = sorted(os.listdir(video_dir))[:600]

for v in videos:
    shutil.copy(
        os.path.join(video_dir, v),
        os.path.join(subset_dir, v)
    )

print("Subset created")

In [ ]:
##checking videos in subset of 600 videos
import os

video_dir = "/content/drive/MyDrive/multimodal_project/dataset/subset_videos"

videos = os.listdir(video_dir)

print("Number of videos:", len(videos))
print(videos[:10])

In [ ]:
#create project strructure
import os

BASE = "/content/drive/MyDrive/multimodal_project"

folders = [
    "frames",
    "audio",
    "image_embeddings",
    "audio_embeddings",
    "video_embeddings",
    "checkpoints",
    "outputs"
]

for f in folders:
    os.makedirs(os.path.join(BASE, f), exist_ok=True)

print("Folders created")

REQUIREMENTS INSTALLATION

In [ ]:
!pip install -q opencv-python
!pip install -q transformers
!pip install -q librosa
!pip install -q moviepy
!pip install -q soundfile

PATHS DEFINING

In [ ]:
#defining paths
import os

BASE_DIR = "/content/drive/MyDrive/multimodal_project"

VIDEO_DIR = os.path.join(
    BASE_DIR,
    "dataset",
    "subset_videos"
)

FRAME_DIR = os.path.join(
    BASE_DIR,
    "frames"
)

AUDIO_DIR = os.path.join(
    BASE_DIR,
    "audio"
)

IMAGE_EMB_DIR = os.path.join(
    BASE_DIR,
    "image_embeddings"
)

AUDIO_EMB_DIR = os.path.join(
    BASE_DIR,
    "audio_embeddings"
)

VIDEO_EMB_DIR = os.path.join(
    BASE_DIR,
    "video_embeddings"
)

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "outputs"
)

print("Video Count:",
      len(os.listdir(VIDEO_DIR)))

EXTRACTING MIDDLE FRAME

In [ ]:
#extract middle frame
import cv2
import os

count = 0

for video in os.listdir(VIDEO_DIR):

    if not video.endswith(".mp4"):
        continue

    video_path = os.path.join(
        VIDEO_DIR,
        video
    )

    cap = cv2.VideoCapture(
        video_path
    )

    total_frames = int(
        cap.get(
            cv2.CAP_PROP_FRAME_COUNT
        )
    )

    middle_frame = total_frames // 2

    cap.set(
        cv2.CAP_PROP_POS_FRAMES,
        middle_frame
    )

    success, frame = cap.read()

    if success:

        frame_name = video.replace(
            ".mp4",
            ".jpg"
        )

        cv2.imwrite(
            os.path.join(
                FRAME_DIR,
                frame_name
            ),
            frame
        )

        count += 1

    cap.release()

print(
    f"Frames Extracted: {count}"
)


In [ ]:
print(
    "Frames:",
    len(os.listdir(FRAME_DIR))
)

EXTRACTING AUDIO

In [ ]:
#extract audio from videos
from moviepy.editor import VideoFileClip
import os

count = 0

for video in os.listdir(VIDEO_DIR):

    if not video.endswith(".mp4"):
        continue

    try:

        video_path = os.path.join(
            VIDEO_DIR,
            video
        )

        clip = VideoFileClip(
            video_path
        )

        audio_path = os.path.join(
            AUDIO_DIR,
            video.replace(
                ".mp4",
                ".wav"
            )
        )

        clip.audio.write_audiofile(
            audio_path,
            verbose=False,
            logger=None
        )

        count += 1

    except:

        pass

print(
    f"Audio Files: {count}"
)

In [ ]:
#failed videos for audio extraction
import os

video_names = {
    f.replace(".mp4", "")
    for f in os.listdir(VIDEO_DIR)
}

audio_names = {
    f.replace(".wav", "")
    for f in os.listdir(AUDIO_DIR)
}

failed = sorted(video_names - audio_names)

print("Failed:", len(failed))
print(failed[:20])

In [ ]:
#verify audios
print(
    "Audio:",
    len(os.listdir(AUDIO_DIR))
)

FRAME EMBEDDINGS

In [ ]:
#image embeddings using clip
from transformers import (
    CLIPProcessor,
    CLIPModel
)

model = CLIPModel.from_pretrained(
    "openai/clip-vit-base-patch32"
)

processor = CLIPProcessor.from_pretrained(
    "openai/clip-vit-base-patch32"
)

print("CLIP Loaded")

In [ ]:
print(type(model))

In [ ]:
from PIL import Image
import numpy as np
import torch
import os

count = 0
failed = []

for image_file in os.listdir(FRAME_DIR):

    try:

        image = Image.open(
            os.path.join(
                FRAME_DIR,
                image_file
            )
        ).convert("RGB")

        inputs = processor(
            images=image,
            return_tensors="pt"
        )

        with torch.no_grad():

            output = model.get_image_features(
                **inputs
            )

        # IMPORTANT FIX
        embedding = output.pooler_output

        np.save(

            os.path.join(
                IMAGE_EMB_DIR,
                image_file.replace(
                    ".jpg",
                    ".npy"
                )
            ),

            embedding.cpu().numpy()
        )

        count += 1

    except Exception as e:

        failed.append(
            (image_file, str(e))
        )

print("Saved:", count)
print("Failed:", len(failed))

if failed:
    print(failed[:5])

In [ ]:
##verify
import os
import numpy as np

sample = os.listdir(
    IMAGE_EMB_DIR
)[0]

emb = np.load(
    os.path.join(
        IMAGE_EMB_DIR,
        sample
    )
)

print("Shape:", emb.shape)

print(
    "Frames:",
    len(os.listdir(FRAME_DIR))
)

print(
    "Image Embeddings:",
    len(os.listdir(IMAGE_EMB_DIR))
)

AUDIO EMBEDDINGS

In [ ]:
#sound enbeddings
!pip install -q librosa soundfile

In [ ]:
import librosa
import numpy as np
import os

count = 0
failed = []

for wav_file in os.listdir(AUDIO_DIR):

    if not wav_file.endswith(".wav"):
        continue

    try:

        audio_path = os.path.join(
            AUDIO_DIR,
            wav_file
        )

        y, sr = librosa.load(
            audio_path,
            sr=16000
        )

        mfcc = librosa.feature.mfcc(
            y=y,
            sr=sr,
            n_mfcc=128
        )

        embedding = np.mean(
            mfcc,
            axis=1
        )

        save_path = os.path.join(
            AUDIO_EMB_DIR,
            wav_file.replace(
                ".wav",
                ".npy"
            )
        )

        np.save(
            save_path,
            embedding
        )

        count += 1

    except Exception as e:

        failed.append(
            (wav_file, str(e))
        )

print("Saved:", count)
print("Failed:", len(failed))

In [ ]:
#verify audio embeddings
print(
    "Audio Embeddings:",
    len(os.listdir(AUDIO_EMB_DIR))
)

sample = os.listdir(
    AUDIO_EMB_DIR
)[0]

emb = np.load(
    os.path.join(
        AUDIO_EMB_DIR,
        sample
    )
)

print(emb.shape)

COMMON LIST FOR VIDEOS WITH BOTH MIDDLE FRAME AND SOUND EMBEDDINGS

In [ ]:
#create common sample list
#image +video embedding
import os

image_ids = {
    f.replace(".npy","")
    for f in os.listdir(
        IMAGE_EMB_DIR
    )
}

audio_ids = {
    f.replace(".npy","")
    for f in os.listdir(
        AUDIO_EMB_DIR
    )
}

common_ids = sorted(
    image_ids.intersection(
        audio_ids
    )
)

print(
    "Common Samples:",
    len(common_ids)
)

In [ ]:
#save common sanmples to csv
import pandas as pd
df = pd.DataFrame({
    "video_id": common_ids
})

csv_path = os.path.join(
    OUTPUT_DIR,
    "common_samples.csv"
)

df.to_csv(
    csv_path,
    index=False
)

print(df.head())


In [ ]:
#Build Dataset Index
import pandas as pd
import os

rows = []

for vid in common_ids:

    rows.append({

        "video_id": vid,

        "image_embedding":
        os.path.join(
            IMAGE_EMB_DIR,
            vid + ".npy"
        ),

        "audio_embedding":
        os.path.join(
            AUDIO_EMB_DIR,
            vid + ".npy"
        )

    })

index_df = pd.DataFrame(rows)

index_df.to_csv(

    os.path.join(
        OUTPUT_DIR,
        "dataset_index.csv"
    ),

    index=False
)

print(index_df.head())

VERIFYING

In [ ]:
print(
    "Videos:",
    len(os.listdir(VIDEO_DIR))
)

print(
    "Frames:",
    len(os.listdir(FRAME_DIR))
)

print(
    "Audio:",
    len(os.listdir(AUDIO_DIR))
)

print(
    "Image Embeddings:",
    len(os.listdir(IMAGE_EMB_DIR))
)

print(
    "Audio Embeddings:",
    len(os.listdir(AUDIO_EMB_DIR))
)

print(
    "Usable Samples:",
    len(common_ids)
)